# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umer6016/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Unit of analysis (Grain):** One row represents one unique piece of content for a specific client on a specific day (Client x Content x Date).
- **Time window:** I am using a mid-panel slice from **March 2026** (`2026-03-01` to `2026-03-31`) to establish historical features, keeping the final month isolated as a sealed test window.
- **Output:** A ranked review queue of content pages that supports human editorial action.

In [8]:
import os
import pandas as pd
from google.colab import userdata
import duckdb

# 1. Ensure we are in the repo directory
if not os.path.exists('flyrank-internship'):
    !git clone https://github.com/umer6016/flyrank-internship.git
if os.getcwd() != '/content/flyrank-internship':
    os.chdir('flyrank-internship')

# 2. Load Hugging Face Token and init DuckDB
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token

    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")

    # --- THE FIX: Explicitly pass the token to DuckDB's secret manager ---
    con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

    base_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
    print("Setup complete. Hugging Face token loaded and DuckDB authenticated for gated data.")
except Exception as e:
    print("ERROR: Please add HF_TOKEN to your Colab Secrets (🔑 icon on the left).")
    print(e)

Setup complete. Hugging Face token loaded and DuckDB authenticated for gated data.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Features (knowable BEFORE the prediction moment):**
  1. `impressions_90d`: Knowable because it aggregates search exposure strictly prior to the cutoff.
  2. `clicks_90d`: Knowable because historical clicks are recorded instantly.
  3. `content_age_days`: Knowable because creation date is static metadata.
  4. `avg_position`: Knowable from historical SERP logs.
  5. `word_count`: Knowable because the page text existed before the decision point.
- **Label / proxy:** `target_is_declining` (derived from `trend_direction`). This is what we predict, never a feature.
- **Context:** `content_id` and `client_id` (used for grouping/joining, never for the model to learn from).
- **Excluded:** I deliberately exclude `health_score` and `priority_score`. Including these product-decision flags would cause a circular result—the model would just memorize the existing rule instead of finding independent signal.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load REAL data for the 5-feature frame
df_real = pd.read_csv("data/raw/content_refresh_anonymized.csv").dropna()
df_real['target_is_declining'] = (df_real['trend_direction'] == 'down').astype(int)

# --- THE LEAKAGE TRAP ---
# We deliberately add a feature that secretly encodes the answer
df_real['LEAKY_future_trend'] = df_real['target_is_declining']
X_leaky = df_real[['impressions_90d', 'clicks_90d', 'content_age_days', 'avg_position', 'word_count', 'LEAKY_future_trend']]
y = df_real['target_is_declining']

model_leaky = DecisionTreeClassifier(max_depth=3).fit(X_leaky, y)
print(f"Score WITH leaky feature: {accuracy_score(y, model_leaky.predict(X_leaky)):.1%} (Suspiciously perfect! The model learned nothing but the leak.)")

# --- REMOVING THE TRAP ---
X_honest = df_real[['impressions_90d', 'clicks_90d', 'content_age_days', 'avg_position', 'word_count']]
model_honest = DecisionTreeClassifier(max_depth=3).fit(X_honest, y)
print(f"Honest score WITHOUT leak: {accuracy_score(y, model_honest.predict(X_honest)):.1%} (Realistic baseline)")


Score WITH leaky feature: 100.0% (Suspiciously perfect! The model learned nothing but the leak.)
Honest score WITHOUT leak: 69.7% (Realistic baseline)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below, we use the strict verification methods from the data contracts guide:
1. `HAVING c > 1` to mathematically prove the grain.
2. `MIN(date)` and `MAX(date)` to prove the window.
3. `AVG(CASE WHEN...)` to check availability/missingness.

In [10]:
print("--- 1. PROVE THE GRAIN (Zero rows returned = Grain holds) ---")
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM '{base_path}'
WHERE report_date = '2026-03-15'
GROUP BY 1, 2, 3
HAVING c > 1
LIMIT 5
"""
display(con.execute(q1).df())

print("\n--- 2. PROVE THE WINDOW & COUNTS ---")
q2 = f"""
SELECT
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(*) as march_row_count
FROM '{base_path}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""
display(con.execute(q2).df())

print("\n--- 3. PROVE MISSINGNESS & AVAILABILITY ---")
q3 = f"""
SELECT
    AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END) as ga4_available_rate,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) as missing_clicks_rate
FROM '{base_path}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""
display(con.execute(q3).df())

--- 1. PROVE THE GRAIN (Zero rows returned = Grain holds) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c



--- 2. PROVE THE WINDOW & COUNTS ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,march_row_count
0,2026-03-01,2026-03-31,9841378



--- 3. PROVE MISSINGNESS & AVAILABILITY ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_available_rate,missing_clicks_rate
0,0.042064,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced History & Absent Data:** History rarely starts together. As seen in the missingness query above, earlier rows often only contain GSC data. This means GA4 engagement data (sessions) isn't necessarily zero—it's completely absent.
- **Macro-Seasonality:** Testing features on just one month makes us blind to yearly seasonality.

In [13]:
print("Checking for unbalanced client history (GSC vs GA4 start dates)...")

q4 = f"""
SELECT
    client_hash_id,
    MIN(report_date) as first_active_date,
    MIN(CASE WHEN ga4_data_available IS TRUE THEN report_date END) as first_ga4_date
FROM '{base_path}'
GROUP BY client_hash_id
HAVING first_ga4_date > first_active_date
LIMIT 5
"""
display(con.execute(q4).df())

print("Limit proven: The clients above have early rows with search data, but their GA4 tracking started much later. We must treat early missing sessions as 'absent, not zero'.")

Checking for unbalanced client history (GSC vs GA4 start dates)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_active_date,first_ga4_date
0,client_9958f0a7ae1df715,2025-01-27,2025-10-29
1,client_73cda7b4e4f265ea,2025-02-11,2026-03-24
2,client_fef1a8f436438636,2025-03-11,2026-03-06
3,client_c182d11e4862a37d,2025-06-21,2026-02-20
4,client_a2eeb8899886adde,2025-07-06,2026-02-19


Limit proven: The clients above have early rows with search data, but their GA4 tracking started much later. We must treat early missing sessions as 'absent, not zero'.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.